# 우선 클래스 500종으로 진행

In [ ]:
import os, time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import convnext_tiny
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm


from pathlib import Path
from PIL import Image

# 경로
SAMPLES_CSV = Path(r"C:/project_sep/csv/samples.csv")
LOG_PATH = Path(r"C:/project_sep/csv/train_log.csv")
MODEL_PATH = Path(r"C:/project_sep/modeling/detection/convnext_tiny_best.pth")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

In [ ]:
df = pd.read_csv(SAMPLES_CSV)


class PillDataset(Dataset):
    def __init__(self, df, split="train", transform=None):
        self.data = df[df["split"] == split].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        path, label = row["image_path"], int(row["class_id"])
        try:
            img = Image.open(path).convert("RGB")
        except:
            img = Image.new("RGB", (224, 224), (255, 255, 255))
        if self.transform:
            img = self.transform(img)
        return img, label

# 간단 EDA

In [ ]:
# 클래스별 샘플 수 상위 10개
print("클래스별 샘플 수 (상위 10):")
print(df["class_id"].value_counts().head(10))

# split별 클래스 개수
print("split별 클래스 개수:")
print(df.groupby("split")["class_id"].nunique())

# dataset 정의

In [ ]:
# class PillDataset(Dataset):
#     def __init__(self, df, split="train", transform=None):
#         self.data = df[df["split"] == split].reset_index(drop=True)
#         self.transform = transform

#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         row = self.data.iloc[idx]
#         path = row["image_path"]
#         label = int(row["class_id"])

#         # 이미지 로드(실패 시 예외 방지)
#         try:
#             img = Image.open(path).convert("RGB")
#         except:
#             # 만약 문제가 있는 파일이면, 흰색 이미지로 대체
#             img = Image.new("RGB", (224, 224), (255, 255, 255))

#         if self.transform:
#             img = self.transform(img)

#         return img, label

# Transform & DataLoader

In [ ]:
# Transform
train_tfms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(8),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)
eval_tfms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

num_classes = df["class_id"].nunique()
print("클래스 수:", num_classes)

train_ds = PillDataset(df, "train", train_tfms)
val_ds = PillDataset(df, "val", eval_tfms)
test_ds = PillDataset(df, "test", eval_tfms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# ConvNeXt Tiny 모델 준비 (pretrained → 분류기 교체)

In [ ]:
model = convnext_tiny(weights="IMAGENET1K_V1")
model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
model = model.to(DEVICE)
print(model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

# 학습/검증 함수

In [ ]:
def run_epoch(model, loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    running_loss, correct = 0.0, 0
    y_true, y_pred = [], []

    loop = tqdm(
        loader, total=len(loader), desc="Train" if train else "Val", leave=False
    )

    for images, labels in loop:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        if train:
            optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        if train:
            loss.backward()
            optimizer.step()

        # 통계
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

        # 진행률 표시 업데이트
        loop.set_postfix(loss=loss.item())

    avg_loss = running_loss / len(loader.dataset)
    acc = correct / len(loader.dataset)
    f1 = f1_score(y_true, y_pred, average="macro")
    return avg_loss, acc, f1

# 학습 루프

In [ ]:
EPOCHS = 5
best_val_acc = 0
log = []

for epoch in range(1, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")

    t0 = time.time()
    train_loss, train_acc, train_f1 = run_epoch(model, train_loader, train=True)
    val_loss, val_acc, val_f1 = run_epoch(model, val_loader, train=False)
    scheduler.step()
    dt = time.time() - t0

    log.append([epoch, train_loss, train_acc, train_f1, val_loss, val_acc, val_f1])
    print(
        f"[{epoch}/{EPOCHS}] "
        f"Train Loss {train_loss:.4f} Acc {train_acc:.3f} F1 {train_f1:.3f} | "
        f"Val Loss {val_loss:.4f} Acc {val_acc:.3f} F1 {val_f1:.3f} | "
        f"{dt:.1f}s"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"  ↳ Best 갱신 (Val Acc={best_val_acc:.3f})")

log_df = pd.DataFrame(
    log,
    columns=[
        "epoch",
        "train_loss",
        "train_acc",
        "train_f1",
        "val_loss",
        "val_acc",
        "val_f1",
    ],
)
log_df.to_csv(LOG_PATH, index=False)
print("로그 저장 완료:", LOG_PATH)

# 체크포인트 저장 함수

In [ ]:
import torch


def save_checkpoint(epoch, model, optimizer, scheduler, best_val_acc, path):
    checkpoint = {
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_val_acc": best_val_acc,
    }
    torch.save(checkpoint, path)
    print(f"체크포인트 저장: {path} (epoch={epoch}, best_val_acc={best_val_acc:.4f})")

# 체크 포인트 불러오기

In [ ]:
def load_checkpoint(path, model, optimizer=None, scheduler=None, device="cuda"):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    if optimizer and scheduler:
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        scheduler.load_state_dict(checkpoint["scheduler_state"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_acc = checkpoint.get("best_val_acc", 0.0)
    print(
        f"체크포인트 로드: {path} (epoch={checkpoint['epoch']}, best_val_acc={best_val_acc:.4f})"
    )
    return model, optimizer, scheduler, start_epoch, best_val_acc

# 학습 루프

In [ ]:
from pathlib import Path

CHECKPOINT_PATH = Path(
    "C:/project_sep/modeling/detection/models/convnext_tiny_ckpt.pth"
)
BEST_PATH = Path("C:/project_sep/modeling/detection/convnext_tiny_best.pth")

EPOCHS = 10  # 총 학습 목표 epoch

start_epoch = 1
best_val_acc = 0.0

# 기존 체크포인트 있으면 불러오기
if CHECKPOINT_PATH.exists():
    model, optimizer, scheduler, start_epoch, best_val_acc = load_checkpoint(
        CHECKPOINT_PATH, model, optimizer, scheduler, device=DEVICE
    )

for epoch in range(start_epoch, EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{EPOCHS} =====")
    train_loss, train_acc, train_f1 = run_epoch(model, train_loader, train=True)
    val_loss, val_acc, val_f1 = run_epoch(model, val_loader, train=False)
    scheduler.step()

    print(f"[{epoch}] Train Acc {train_acc:.3f} | Val Acc {val_acc:.3f}")

    # Best 모델 따로 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), BEST_PATH)
        print(f"  ↳ Best 갱신 (Val Acc={best_val_acc:.3f})")

    # 매 epoch마다 체크포인트 저장 (resume 가능)
    save_checkpoint(epoch, model, optimizer, scheduler, best_val_acc, CHECKPOINT_PATH)

# 테스트 성능 + 상세 리포트

In [ ]:
# 베스트 모델 로드
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

test_loss, test_acc, test_f1 = run_epoch(model, test_loader, train=False)
print(f"[TEST] Loss {test_loss:.4f} | Acc {test_acc:.3f} | F1 {test_f1:.3f}")

# classification report (상위 20개 클래스만 표시)
from sklearn.metrics import classification_report

y_true, y_pred = [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        preds = outputs.argmax(1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

report = classification_report(y_true, y_pred, output_dict=False, digits=3)
print(report)